# Análisis exploratorio: BAF Base

**Objetivo.** Conocer los datos antes de modelar y fijar las decisiones de
preprocesamiento con evidencia. Las conclusiones se trasladan a
`docs/datos.md` y a la memoria del TFM.

**Contenido**
1. Carga de los datos
2. Prevalencia de fraude por mes y por partición
3. Valores ausentes: ¿la ausencia se relaciona con el fraude?
4. Deriva entre meses
5. Atributo protegido (`customer_age >= 50`)

## 1. Carga de los datos

Se carga `Base.csv` con las rutas de `configs/local.yaml`. `resolve_path`
convierte la ruta en absoluta desde la raíz del repositorio, para que funcione
aunque el notebook se ejecute desde `notebooks/`.

In [ ]:
import pandas as pd

from src.config import load_config, resolve_path
from src.data import load_data, MISSING_AS_MINUS_ONE

config = load_config(resolve_path("configs/local.yaml"))
df = load_data(config)

df.shape


(1000000, 32)

**Resultado.** 1.000.000 de filas y 32 columnas, como indica la documentación
del BAF. La verificación de tipos, rangos y ausentes codificados como negativos
está en `docs/datos.md`.

## 2. Prevalencia de fraude por mes y por partición

Partición fija del proyecto: entrenamiento = meses 0–4, validación = mes 5,
test = meses 6–7.

**Qué se mira.** El número de solicitudes y la tasa de fraude en cada mes y en
cada partición.

**Por qué importa.** El umbral se fija en validación (5 % de FPR) y se aplica
tal cual al test. Si la distribución cambia entre el mes 5 y los meses 6–7, ese
umbral no dará exactamente un 5 % de FPR en test.

In [5]:
import numpy as np

""""
def particionar_df(x):
    if x <= 4:
        return "train"
    elif x == 5:
        return "eval"
    elif 5 < x <=7:
        return "test"
    else:
        return 0

df["split"] = df["month"].apply(particionar_df)


train = df[df["split"].str.contains("train")].copy().reset_index(drop=True)
eval = df[df["split"].str.contains("eval")].copy().reset_index(drop=True)
test = df[df["split"].str.contains("test")].copy().reset_index(drop=True)

print(len(train))

"""
conditions = [df["month"] <=4, df["month"] ==5, df["month"] >= 6]
df["split"] = np.select(conditions, ["train", "val", "test"], default = "unknown")

assert(df["split"] != "unknown").all(), "Hay meses fuera de 0-7"

df.groupby("split")["fraud_bool"].agg(["size", "sum", "mean"])



,size,sum,mean
split,,,
test,205011,2878,0.014038
train,675666,6740,0.009975
val,119323,1411,0.011825


In [4]:
df.groupby("month")["fraud_bool"].agg(["size", "sum", "mean"])

,size,sum,mean
month,,,
0,132440,1500,0.011326
1,127620,1198,0.009387
2,136979,1198,0.008746
3,150936,1392,0.009222
4,127691,1452,0.011371
5,119323,1411,0.011825
6,108168,1450,0.013405
7,96843,1428,0.014746


## 3. Valores ausentes: ¿la ausencia se relaciona con el fraude?

**Qué se mira.**
- En las columnas con -1 como ausente, la tasa de fraude con y sin el dato.
- En `intended_balcon_amount`, la tasa de fraude por tramos de los valores
  negativos, para decidir si son ausentes (decisión pendiente en
  `docs/datos.md`).

**Por qué importa.** Si la ausencia predice el fraude, el indicador de
ausencia que usarán las redes aporta información.

In [12]:
df_minus_one = df[MISSING_AS_MINUS_ONE]

assert(df_minus_one >= -1).all().all(), "Hay valores distintos de -1"

In [15]:
rows = []

for col in MISSING_AS_MINUS_ONE:
    is_missing = (df[col] == -1)
    rows.append({
        "column": col,
        "n_missing": is_missing.sum(),
        "pct_missing": is_missing.mean(),
        "fraud_rate_missing": df.loc[is_missing, "fraud_bool"].mean(),
        "fraud_rate_present": df.loc[~is_missing, "fraud_bool"].mean(),
        "ratio": df.loc[is_missing, "fraud_bool"].mean() / df.loc[~is_missing, "fraud_bool"].mean()
    })


summary = pd.DataFrame(rows)
summary

,column,n_missing,pct_missing,fraud_rate_missing,fraud_rate_present,ratio
0,prev_address_months_count,712920,0.712920,0.014215,0.003118,4.559529
1,current_address_months_count,4254,0.004254,0.003291,0.011062,0.297505
2,bank_months_count,253635,0.253635,0.016323,0.009230,1.768423
3,session_length_in_minutes,2015,0.002015,0.008933,0.011033,0.809645
4,device_distinct_emails_8w,359,0.000359,0.011142,0.011029,1.010255


In [ ]:
# intended_balcon_amount: solo train y val, porque de aquí sale una decisión
# de preprocesado y el test no debe influir en ella
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
col = "intended_balcon_amount"

# Quintiles de los negativos, con los bordes fijados en train
_, edges = pd.qcut(train.loc[train[col] < 0, col], 5, retbins=True)
edges[0], edges[-1] = -np.inf, 0

for name, data in [("train", train), ("val", val)]:
    is_negative = data[col] < 0
    rate_negative = data.loc[is_negative, "fraud_bool"].mean()
    rate_rest = data.loc[~is_negative, "fraud_bool"].mean()
    print(f"{name}: fraude con negativo {rate_negative:.2%}, resto {rate_rest:.2%}, ratio {rate_negative / rate_rest:.2f}")

    bins = pd.cut(data.loc[is_negative, col], edges, right=False)
    display(data[is_negative].groupby(bins, observed=True)["fraud_bool"].agg(["size", "sum", "mean"]).round(4))

# Forma de la distribución de los negativos
train.loc[train[col] < 0, col].describe(percentiles=[0.01, 0.5, 0.99]).round(3)

**`intended_balcon_amount`.** El datasheet oficial del BAF lo aclara: rango
[-16, 114], *"negatives are missing values"*. La descripción de Kaggle da otro
rango y no lo menciona.

- **Tener un negativo sí se relaciona con el fraude**: 1,19 % frente a 0,48 % en
  train (ratio 2,46) y 1,41 % frente a 0,42 % en val (ratio 3,35).
- **La magnitud del negativo aporta mucho menos.** Entre el primer y el último
  quintil de los negativos, la tasa pasa del 1,02 % al 1,33 % en train (ratio 1,31)
  y del 1,18 % al 1,75 % en val (ratio 1,47). Va en la misma dirección en ambas
  particiones, pero no en todos los meses: en los meses 1 y 2 de train es casi
  plana.
- **Los negativos se concentran alrededor de -1** (mediana -1,00; percentiles 1 y
  99: -1,86 y -0,17). Podría ser un -1 original más el ruido que, según el
  datasheet, se añadió a los datos antes de generarlos con una GAN. Es una
  hipótesis, no algo que el datasheet afirme.

**Recomendación (pendiente de aprobar).** Tratar los negativos como ausentes,
igual que el resto: convertirlos a NaN y conservar la señal principal con el
indicador de ausencia. Se pierde el gradiente dentro de los negativos, que es débil
y que no tiene una interpretación real si esos valores significan "sin dato".

## 4. Deriva entre meses

**Qué se mira.** Si la distribución de las variables cambia de un mes a otro,
sobre todo entre entrenamiento (0–4) y test (6–7).

**Por qué importa.** El modelo se entrena con el pasado y se evalúa con el
futuro. Una deriva fuerte explicaría caídas de rendimiento en test.

In [51]:
# Variables de entrada numéricas: sin el objetivo, el mes y la constante
num_cols = df.select_dtypes(include="number").columns.drop(
    ["fraud_bool", "month", "device_fraud_count"]
)

df.groupby("month")[num_cols].mean()

,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,velocity_24h,...,email_is_free,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,session_length_in_minutes,keep_alive_session,device_distinct_emails_8w
month,,,,,,,,,,,,,,,,,,,,,
0,0.545999,0.480271,17.995500,85.359333,33.566898,0.721744,10.320819,2168.830384,7436.980252,5882.409562,...,0.571549,0.412647,0.896927,12.224049,0.170757,591.471685,0.039709,8.216170,0.677582,1.025619
1,0.520698,0.526254,13.645886,100.431257,33.093637,1.346258,8.377181,1676.173719,6821.410104,5544.745790,...,0.587870,0.423061,0.859959,10.662592,0.208776,522.967247,0.020757,8.092347,0.590644,1.034117
2,0.496552,0.536123,18.444623,80.539418,33.664503,1.511642,9.040804,1713.588492,6465.410441,5293.535446,...,0.528796,0.360442,0.872083,10.405215,0.175195,451.502055,0.022281,8.844311,0.559867,1.034159
3,0.563151,0.485859,20.081637,79.237511,34.426711,0.996418,10.338422,1541.099744,5960.157365,4972.625369,...,0.552320,0.346465,0.889947,10.503180,0.272321,589.499324,0.022685,7.473184,0.520466,1.009706
4,0.580703,0.490350,16.521697,90.412261,33.537916,1.141969,7.247711,1561.108817,5323.332006,4612.584795,...,0.490144,0.400091,0.880493,10.702712,0.244301,507.618235,0.026556,7.038164,0.480512,1.010635
5,0.580014,0.473809,9.938302,95.400208,33.557068,0.639925,7.723208,1418.399479,4755.840267,4218.687532,...,0.517394,0.506558,0.912816,11.089497,0.274147,478.774168,0.024203,6.593297,0.592484,1.010769
6,0.626423,0.462752,15.095010,89.784437,34.417388,0.860065,9.575181,1295.843928,3958.599974,3691.506701,...,0.470463,0.447452,0.913819,10.970990,0.240210,489.143370,0.022983,7.039296,0.570945,1.008884
7,0.617458,0.484813,21.765652,70.565389,33.074461,0.884328,5.616526,985.613013,3605.322602,3260.626953,...,0.497238,0.483628,0.900013,10.041035,0.193488,465.658230,0.021612,6.585083,0.648173,1.008436


In [52]:
# Proporción de cada categoría por mes (cada fila suma 1)
cat_cols = df.select_dtypes(exclude="number").columns.drop("split")

for col in cat_cols:
    display(pd.crosstab(df["month"], df[col], normalize="index").round(3))

payment_type,AA,AB,AC,AD,AE
month,,,,,
0,0.310,0.338,0.138,0.214,0.0
1,0.257,0.340,0.285,0.118,0.0
2,0.273,0.308,0.315,0.103,0.0
3,0.287,0.361,0.255,0.097,0.0
4,0.209,0.396,0.276,0.118,0.0
5,0.231,0.442,0.227,0.099,0.0
6,0.288,0.385,0.243,0.085,0.0
7,0.188,0.421,0.281,0.110,0.0


employment_status,CA,CB,CC,CD,CE,CF,CG
month,,,,,,,
0,0.648,0.223,0.034,0.029,0.021,0.045,0.000
1,0.695,0.167,0.037,0.031,0.031,0.039,0.000
2,0.739,0.131,0.040,0.027,0.020,0.043,0.000
3,0.731,0.128,0.048,0.028,0.014,0.049,0.001
4,0.741,0.110,0.045,0.031,0.024,0.049,0.001
5,0.761,0.111,0.030,0.019,0.030,0.048,0.000
6,0.775,0.107,0.039,0.019,0.021,0.039,0.000
7,0.773,0.116,0.023,0.026,0.023,0.037,0.000


housing_status,BA,BB,BC,BD,BE,BF,BG
month,,,,,,,
0,0.168,0.180,0.418,0.031,0.200,0.002,0.001
1,0.138,0.282,0.329,0.020,0.229,0.002,0.000
2,0.109,0.313,0.401,0.016,0.159,0.002,0.000
3,0.195,0.251,0.427,0.017,0.108,0.001,0.000
4,0.205,0.244,0.361,0.020,0.168,0.002,0.000
5,0.162,0.276,0.327,0.033,0.201,0.001,0.000
6,0.192,0.307,0.303,0.034,0.163,0.002,0.000
7,0.198,0.240,0.387,0.046,0.127,0.001,0.000


source,INTERNET,TELEAPP
month,,
0,0.994,0.006
1,0.992,0.008
2,0.995,0.005
3,0.992,0.008
4,0.990,0.010
5,0.994,0.006
6,0.991,0.009
7,0.996,0.004


device_os,linux,macintosh,other,windows,x11
month,,,,,
0,0.328,0.040,0.382,0.242,0.008
1,0.377,0.045,0.298,0.274,0.005
2,0.396,0.047,0.281,0.269,0.007
3,0.324,0.060,0.340,0.268,0.008
4,0.293,0.072,0.328,0.299,0.008
5,0.297,0.060,0.379,0.256,0.008
6,0.291,0.054,0.412,0.235,0.008
7,0.347,0.051,0.338,0.260,0.004


In [53]:
def psi(reference, actual, n_bins=10, max_discrete=20, eps=1e-6):
    """Population Stability Index de `actual` respecto a `reference`.

    - Variables con pocos valores distintos (binarias, categóricas, discretas):
      un tramo por valor.
    - Variables continuas: tramos por cuantiles de la referencia.
    """
    reference = pd.Series(reference)
    actual = pd.Series(actual)

    if not pd.api.types.is_numeric_dtype(reference) or reference.nunique() <= max_discrete:
        # Un tramo por valor; se incluyen los valores que solo aparecen en `actual`
        values = reference.value_counts().index.union(actual.value_counts().index)
        p_ref = reference.value_counts(normalize=True).reindex(values, fill_value=0).to_numpy()
        p_act = actual.value_counts(normalize=True).reindex(values, fill_value=0).to_numpy()
    else:
        # Bordes de los tramos: cuantiles de la referencia, con extremos abiertos
        edges = np.unique(np.quantile(reference, np.linspace(0, 1, n_bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        p_ref = np.histogram(reference, bins=edges)[0] / len(reference)
        p_act = np.histogram(actual, bins=edges)[0] / len(actual)

    # Evitar divisiones por 0 y log(0)
    p_ref = np.clip(p_ref, eps, None)
    p_act = np.clip(p_act, eps, None)

    return np.sum((p_act - p_ref) * np.log(p_act / p_ref))

In [54]:
# Comprobación de psi() con casos de resultado conocido
rng = np.random.default_rng(0)
a = rng.normal(0, 1, 100_000)

print("Continua, misma distribución: ", round(psi(a, rng.normal(0, 1, 100_000)), 4))    # ≈ 0
print("Continua, desplazada 0,5:     ", round(psi(a, rng.normal(0.5, 1, 100_000)), 4))  # > 0,1
print("Binaria, misma proporción:    ", round(psi(rng.binomial(1, 0.3, 100_000), rng.binomial(1, 0.3, 100_000)), 4))  # ≈ 0
print("Binaria, de 0,3 a 0,5:        ", round(psi(rng.binomial(1, 0.3, 100_000), rng.binomial(1, 0.5, 100_000)), 4))  # > 0,1

Continua, misma distribución:  0.0003
Continua, desplazada 0,5:      0.2391
Binaria, misma proporción:     0.0
Binaria, de 0,3 a 0,5:         0.174


In [55]:
# PSI de test (meses 6–7) respecto a train (meses 0–4), numéricas y categóricas
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

rows = []
for col in [*num_cols, *cat_cols]:
    rows.append({"column": col, "psi": psi(train[col], test[col])})

psi_table = pd.DataFrame(rows).sort_values("psi", ascending=False).round(4)
psi_table

,column,psi
10,velocity_4w,3.7388
9,velocity_24h,1.8918
8,velocity_6h,1.0886
7,zip_count_4w,0.4909
12,date_of_birth_distinct_emails_4w,0.3989
13,credit_risk_score,0.2612
19,proposed_credit_limit,0.0930
0,income,0.0820
21,session_length_in_minutes,0.0625
1,name_email_similarity,0.0438


**Conclusión.** Seis variables tienen deriva fuerte (PSI > 0,25) entre train y
test: `velocity_4w` (3,74), `velocity_24h` (1,89), `velocity_6h` (1,09),
`zip_count_4w` (0,49), `date_of_birth_distinct_emails_4w` (0,40) y
`credit_risk_score` (0,26). `proposed_credit_limit` (0,093) e `income` (0,082)
quedan cerca del umbral de 0,1. Las binarias y categóricas son estables
(PSI < 0,03).

- **La deriva es un desplazamiento dentro del rango, no valores nuevos.** Solo un
  0,011 % o menos del test queda fuera del rango de train. Pero el 88 % del test
  en `velocity_4w` está por debajo del percentil 10 de train: los modelos verán
  sobre todo valores que en train eran poco frecuentes.
- **Las `velocity` bajan todos los meses**: `velocity_6h` pasa de 7.437 a 3.605 de
  media. El volumen de solicitudes no lo explica, porque el máximo está en el mes
  3 (150.936) y la velocidad sigue bajando. La causa no se conoce.
- **`credit_risk_score` se comporta de otra forma**: no baja de forma continua,
  sino que sube de nivel en el mes 3 (media 109–117 en los meses 0–2, 137–148
  después).

**Implicación.** El mes 5 tampoco coincide con el test (`velocity_6h` media 4.756
frente a 3.605–3.959), así que el FPR real en test puede alejarse del 5 % fijado
en validación. Hay que vigilarlo con el "FPR real en test" ya previsto en las
métricas. El PSI solo mide la distribución de cada variable: no dice si cambia su
relación con el fraude.

## 5. Atributo protegido: edad

Grupos: `customer_age >= 50` frente a `< 50`, el corte del código oficial del
BAF (ver `docs/datos.md`).

**Qué se mira.** El tamaño de cada grupo y su tasa de fraude, en total y por
partición.

**Por qué importa.** La métrica de equidad es el cociente de FPR entre los
dos grupos. Un grupo pequeño en test da un FPR con mucha varianza.

In [ ]:
# Serie aparte, no columna de df, para no añadirla a las categóricas de la sección 4
age_group = (df["customer_age"] >= 50).map({True: ">=50", False: "<50"}).rename("age_group")

# Tamaño y tasa de fraude de cada grupo, en total y por partición
display(df.groupby(age_group)["fraud_bool"].agg(["size", "sum", "mean"]).round(4))
display(df.groupby([df["split"], age_group])["fraud_bool"].agg(["size", "sum", "mean"]).round(4))

# Detalle por tramo de edad
display(df.groupby("customer_age")["fraud_bool"].agg(["size", "sum", "mean"]).round(4))

# Proporción del grupo >=50 por mes
display((age_group == ">=50").groupby(df["month"]).mean().round(3))

# Legítimos por grupo: son la base sobre la que se calcula el FPR
legit = df["fraud_bool"] == 0
df[legit].groupby([df["split"], age_group]).size().unstack()

**Conclusión.** El grupo `>= 50` es el 18,3 % de las solicitudes (183.013) y su
tasa de fraude es casi el triple: 2,34 % frente a 0,83 % (ratio 2,84). La relación
se mantiene en las tres particiones (ratio 3,07 en train, 2,38 en val y 2,65 en
test) y la tasa crece con cada tramo de edad, desde el 0,35 % en el tramo 10 hasta
el 4,93 % en el 80. El tramo 90 tiene solo 76 filas.

- **El tamaño del grupo no es un problema para medir el FPR.** En test hay 32.790
  legítimos de 50 años o más. Con un FPR del 5 %, el error estándar es de unos
  0,12 puntos porcentuales.
- **El grupo pierde peso en el mes 7**: 13,1 %, frente a un 17–20 % en el resto de
  meses.

**Implicación.** Como el fraude es más frecuente en el grupo mayor, cabe esperar
que los modelos le asignen puntuaciones más altas y un FPR mayor con un umbral
único. Es una hipótesis que se comprobará con el cociente de FPR de cada modelo.